In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv('IMDB Dataset.csv')

In [8]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [9]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


### **Text Preprocessing**

In [11]:
# 10k sample rows out of 50k

df = df.sample(10000)
df = df.reset_index(drop=True)
df.shape

(10000, 2)

In [12]:
# Label Encoding
df.replace({'positive':1, 'negative':0}, inplace=True)

/var/folders/gz/lp_tdb7s50v1gdw2n4d53qq80000gn/T/ipykernel_53601/1142663475.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'positive':1, 'negative':0}, inplace=True)


In [13]:
# Eliminate HTML Tags
from bs4 import BeautifulSoup
df['review'] = df['review'].apply(
    lambda x: BeautifulSoup(str(x), "html.parser").get_text()
)

In [14]:
# Convert to lowercase
df['review'] = df['review'].str.lower()

In [15]:
# Eliminate Special Characters
import re
df['review'] = df['review'].apply(
    lambda x: re.sub(r"[^a-z0-9\s]", "", x)
)

In [16]:
# Eliminate Stop Words
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tanjimrahman/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
  words = text.split()
  new_words = []
  for word in words:
    if word not in stop_words:
      new_words.append(word)
  return new_words

df['review'] = df['review'].apply(remove_stopwords)


In [18]:
# Stemming Words
from nltk.stem import PorterStemmer
ps = PorterStemmer()

def stem_text(text):
    stemmed_words = []
    for word in text:
        stemmed_words.append(ps.stem(word))
    return " ".join(stemmed_words)

df['review'] = df['review'].apply(stem_text)

In [19]:
df

,review,sentiment
0,subject movi disturb could otherwis intellig r...,1
1,cannot believ movi ever creat think point dire...,0
2,film never receiv attent deserv although one f...,1
3,barney idiot dinosaur unfortunalt didnt go ext...,0
4,liv tayler sexiest movi incorpor femm fatal ro...,1
...,...,...
9995,pegg hit past year start shaun dead 2004 movi ...,1
9996,stoic lacon soldier sergeant todd fine credibl...,1
9997,sometim 1998 saban acquir right produc brandne...,0
9998,okay come long way houston whenev see movi tak...,1


### **Vectorization**

In [20]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [21]:
X = cv.fit_transform(df['review']).toarray()
y = df.iloc[:, -1].values

In [22]:
X.shape

(10000, 62566)

In [23]:
y.shape

(10000,)

### **Model Train**

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 62566)
(2000, 62566)
(8000,)
(2000,)


In [26]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

nb1 = GaussianNB()
nb2 = MultinomialNB()
nb3 = BernoulliNB()

In [27]:
nb1.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


In [28]:
nb2.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [29]:
nb3.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [30]:
y_pred1 = nb1.predict(X_test)
y_pred2 = nb2.predict(X_test)
y_pred3 = nb3.predict(X_test)

### **Model Test**

In [31]:
from sklearn.metrics import classification_report

In [33]:
# Gaussian Naive Bayes
report1 = classification_report(y_test, y_pred1)
print(report1)

              precision    recall  f1-score   support

           0       0.61      0.75      0.67       997
           1       0.67      0.52      0.59      1003

    accuracy                           0.63      2000
   macro avg       0.64      0.63      0.63      2000
weighted avg       0.64      0.63      0.63      2000



In [34]:
# Multinomial Naive Bayes
report2 = classification_report(y_test, y_pred2)
print(report2)

              precision    recall  f1-score   support

           0       0.83      0.87      0.85       997
           1       0.86      0.82      0.84      1003

    accuracy                           0.84      2000
   macro avg       0.85      0.84      0.84      2000
weighted avg       0.85      0.84      0.84      2000



In [35]:
# Bernoulli Naive Bayes
report3 = classification_report(y_test, y_pred3)
print(report3)

              precision    recall  f1-score   support

           0       0.81      0.88      0.84       997
           1       0.87      0.80      0.83      1003

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000

